# Escalamiento: evidencia numérica para elegir

## Pregunta y protocolo
¿Qué cambia al usar ninguna transformación, StandardScaler o MinMaxScaler en KNN, regresión logística, SVM y árbol? Usamos Wine y los mismos cinco pliegues para las 12 combinaciones. Elegimos por F1 macro medio de CV; el test no participa. Esta es una comparación controlada con hiperparámetros fijos, no una búsqueda exhaustiva del mejor modelo posible.

| Método | Fórmula por columna | Interpretación |
|---|---|---|
| StandardScaler | (x − media) / desviación | Centro 0 y varianza 1 en entrenamiento no constante |
| MinMaxScaler | (x − mínimo) / (máximo − mínimo) | Rango observado [0,1] en entrenamiento |
| Sin escala | x | Conserva unidades originales |

StandardScaler no vuelve normal una distribución. Ambos son sensibles a extremos; MinMax puede comprimir la mayoría de datos. Los valores nuevos pueden salir de [0,1]. Los árboles suelen ser casi invariantes a estas transformaciones monótonas; diferencias pequeñas pueden venir de empates o precisión numérica. En modelos regularizados, fijar C al cambiar escala también cambia la penalización efectiva: se debe reconocer al interpretar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.base import clone
datos = load_wine(as_frame=True)
X, y = datos.data, datos.target
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
cv = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X_dev, y_dev))
display(X_dev.describe().T[['mean','std','min','max']])
display(y_dev.value_counts().sort_index().rename('Casos por clase').to_frame())

## Cálculo manual y comprobación

In [ ]:
a=np.array([10.,20.,30.]).reshape(-1,1)
ss=StandardScaler().fit(a); mm=MinMaxScaler().fit(a)
display(pd.DataFrame({'original':a.ravel(),'StandardScaler':ss.transform(a).ravel(),'MinMaxScaler':mm.transform(a).ravel()}))
assert np.allclose(ss.transform(a).ravel(),(a.ravel()-a.mean())/a.std(ddof=0))
assert np.allclose(mm.transform(a).ravel(),[0,.5,1])
print('Nuevo valor 40:',ss.transform([[40]])[0,0],mm.transform([[40]])[0,0])
# Contribuciones a la distancia entre dos casos de desarrollo.
distancias={}
for nombre,esc in [('Original',None),('Standard',StandardScaler()),('MinMax',MinMaxScaler())]:
    z=X_dev.to_numpy() if esc is None else esc.fit_transform(X_dev)
    aportes=(z[0]-z[1])**2
    distancias[nombre]=100*aportes/aportes.sum()
aportes=pd.DataFrame(distancias,index=X.columns)
display(aportes)
aportes.plot.bar(figsize=(12,4),ylabel='% de distancia cuadrada',title='Aporte por variable: dos casos de desarrollo'); plt.tight_layout(); plt.show()

## Comparación justa
El escalador de la demostración anterior no se reutiliza. Cada pipeline aprende una transformación nueva dentro del entrenamiento de cada pliegue. La desviación entre pliegues describe variabilidad, no significancia estadística.

In [ ]:
from sklearn.svm import SVC
modelos={'KNN':KNeighborsClassifier(n_neighbors=5),'Logística':LogisticRegression(C=1,max_iter=10000,solver='newton-cholesky'),'SVM':SVC(C=1,gamma='scale'),'Árbol':DecisionTreeClassifier(max_depth=4,random_state=42)}
escalas={'Sin escala':'passthrough','StandardScaler':StandardScaler(),'MinMaxScaler':MinMaxScaler()}
filas=[]; pipelines={}
for nombre,modelo in modelos.items():
    for escala,transformacion in escalas.items():
        paso=transformacion if isinstance(transformacion,str) else clone(transformacion)
        pipe=Pipeline([('escala',paso),('modelo',clone(modelo))])
        resultado=cross_validate(pipe,X_dev,y_dev,cv=cv,scoring='f1_macro',return_train_score=True,error_score='raise',n_jobs=1)
        fila={'modelo':nombre,'escala':escala,'F1_CV':resultado['test_score'].mean(),'desviacion_CV':resultado['test_score'].std(),'F1_train':resultado['train_score'].mean()}
        fila.update({f'pliegue_{i+1}':v for i,v in enumerate(resultado['test_score'])})
        filas.append(fila); pipelines[(nombre,escala)]=pipe
tabla=pd.DataFrame(filas)
display(tabla.round(4))
assert np.isfinite(tabla.select_dtypes('number')).all().all()
tabla.pivot(index='modelo',columns='escala',values='F1_CV').plot.bar(ylim=(0,1),ylabel='F1 macro medio de CV',title='Efecto del escalamiento con hiperparámetros fijos',figsize=(10,4)); plt.tight_layout(); plt.show()

## Justificación numérica calculada
Se informa la mejor media y la mejora frente a no escalar. Las diferencias pequeñas o variables entre pliegues no demuestran superioridad general. En empates exactos, el orden de la tabla prioriza sin escala, luego StandardScaler y luego MinMaxScaler.

In [ ]:
for nombre in modelos:
    bloque=tabla[tabla.modelo==nombre]
    mejor=bloque.loc[bloque.F1_CV.idxmax()]
    base=bloque[bloque.escala=='Sin escala'].iloc[0]
    cols=[f'pliegue_{i}' for i in range(1,6)]
    delta=mejor[cols].to_numpy(dtype=float)-base[cols].to_numpy(dtype=float)
    print(f"{nombre}: elegir {mejor.escala} en este experimento; F1 CV={mejor.F1_CV:.4f}, desviación={mejor.desviacion_CV:.4f}; diferencia frente a sin escala={mejor.F1_CV-base.F1_CV:+.4f}.")
    print('Diferencias emparejadas por pliegue:',np.round(delta,4))
elegido=tabla.loc[tabla.F1_CV.idxmax()]
final=clone(pipelines[(elegido.modelo,elegido.escala)]).fit(X_dev,y_dev)
print('Combinación global elegida antes de test:',elegido.modelo,elegido.escala)

## Evaluación final
Solo la combinación global seleccionada se evalúa en test. No se usan doce resultados de test para volver a elegir.

In [ ]:
pred=final.predict(X_test)
print(classification_report(y_test,pred,target_names=datos.target_names,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=datos.target_names); plt.title('Test reservado: combinación elegida'); plt.show()

## Conclusiones que debe redactar
1. Identifique qué variable dominó la distancia original y qué cambió al escalar.
2. Cite media, desviación y diferencias por pliegue para justificar cada elección; no escriba «StandardScaler siempre es mejor».
3. Explique por qué el árbol responde de modo distinto a KNN.
4. Proponga una extensión con GridSearchCV que ajuste conjuntamente escalador y C o k, usando solo desarrollo.
5. Investigue extremos y considere RobustScaler en una extensión, sin borrar observaciones automáticamente.

Fuente: [scikit-learn, comparación de escaladores](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html).

## Galería de rendimiento y clasificación

Ejecute estas celdas después de las anteriores. Las imágenes y gráficos se generan con las predicciones reales de esta ejecución. El test se usa solo para diagnóstico del modelo ya seleccionado. No ajuste parámetros después de observar sus errores.

In [ ]:
modelo_visual = final

from sklearn.metrics import precision_recall_fscore_support, accuracy_score
# Diagnóstico del modelo ya elegido. No seleccionar otra vez mirando test.
hay_probabilidades = hasattr(modelo_visual,'predict_proba')
prob = modelo_visual.predict_proba(X_test) if hay_probabilidades else modelo_visual.decision_function(X_test)
tipo_salida = 'Probabilidad estimada' if hay_probabilidades else 'Puntuación de decisión (no es probabilidad)'
clases = modelo_visual.classes_
pred_visual = modelo_visual.predict(X_test)
precision, recall, f1, soporte = precision_recall_fscore_support(y_test,pred_visual,labels=clases,zero_division=0)
metricas = pd.DataFrame({'Precisión':precision,'Recall':recall,'F1':f1},index=datos.target_names)
display(metricas.assign(Casos=soporte))
fig, axes = plt.subplots(1,2,figsize=(12,4))
metricas.plot.bar(ax=axes[0],ylim=(0,1.08),rot=0)
axes[0].set(title='Rendimiento por clase en test',ylabel='Métrica')
correctos = pred_visual == np.asarray(y_test)
axes[1].bar(['Aciertos','Errores'],[correctos.sum(),(~correctos).sum()],color=['#08796e','#b05220'])
axes[1].set(title=f'Exactitud en test: {accuracy_score(y_test,pred_visual):.3f}',ylabel='Número de casos')
plt.tight_layout(); plt.show()

# Cada fila es un caso; los errores se muestran primero sin alterar el modelo.
orden = np.argsort(correctos,kind='stable')[:15]
fig, ax = plt.subplots(figsize=(8,6))
im=ax.imshow(prob[orden],vmin=0 if hay_probabilidades else prob.min(),vmax=1 if hay_probabilidades else prob.max(),cmap='Blues',aspect='auto')
ax.set_xticks(range(len(clases)),datos.target_names)
ax.set_yticks(range(len(orden)),[f'Caso {i}: real {np.asarray(y_test)[i]}, pred. {pred_visual[i]}' for i in orden])
for fila,i in enumerate(orden):
    for col in range(len(clases)):
        ax.text(col,fila,f'{prob[i,col]:.2f}',ha='center',va='center',color='white' if im.norm(prob[i,col])>.6 else 'black')
ax.set(title=tipo_salida+'; hasta 15 casos de test, errores primero',xlabel='Clase')
fig.colorbar(im,ax=ax,label=tipo_salida); plt.tight_layout(); plt.show()
if hay_probabilidades: assert np.allclose(prob.sum(axis=1),1)


In [ ]:
# Mostrar cada pliegue, además de la media: no son intervalos de confianza.
fig,axes=plt.subplots(2,2,figsize=(12,8),sharey=True)
cols=[f'pliegue_{i}' for i in range(1,6)]
for ax,nombre in zip(axes.ravel(),modelos):
    bloque=tabla[tabla.modelo==nombre]
    for _,fila in bloque.iterrows():
        ax.plot(range(1,6),fila[cols].to_numpy(dtype=float),marker='o',label=fila.escala)
    ax.set(title=nombre,xlabel='Pliegue',ylabel='F1 macro de validación',ylim=(0,1.05),xticks=range(1,6)); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
fig,axes=plt.subplots(1,2,figsize=(12,4))
resumen=tabla.pivot(index='modelo',columns='escala',values='F1_CV')
im=axes[0].imshow(resumen,vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
axes[0].set_xticks(range(3),resumen.columns,rotation=20); axes[0].set_yticks(range(4),resumen.index)
for i in range(4):
    for j in range(3): axes[0].text(j,i,f'{resumen.iloc[i,j]:.3f}',ha='center',va='center',color='white' if resumen.iloc[i,j]>.65 else 'black')
axes[0].set_title('F1 medio: comparación de 12 combinaciones'); fig.colorbar(im,ax=axes[0])
brecha=tabla.assign(brecha=tabla.F1_train-tabla.F1_CV).pivot(index='modelo',columns='escala',values='brecha')
brecha.plot.bar(ax=axes[1],rot=0); axes[1].axhline(0,color='black',linewidth=.7); axes[1].set(title='Brecha entre entrenamiento y validación',ylabel='F1 train − F1 CV')
plt.tight_layout(); plt.show()

### Interprete las figuras
Identifique las clases más difíciles, describa dos patrones visibles y distinga los gráficos de validación de los de test. Si propone un cambio del modelo, deberá evaluarlo con una nueva separación independiente; no convierta el test observado en un conjunto de selección.